In [65]:
import pandas as pd
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler as scale
from sklearn.decomposition import PCA

In [40]:
foods = pd.read_csv('../02_Data_formatted/0201_data/nv_categorized.csv', 
                    index_col = 'food')

In [125]:
# protein composition
protein_aminos = pd.concat([foods.loc[(foods.protein>6) & (foods.protein*4/foods.calories>.2),"protein"],foods.filter(
    like = "aa",axis=1)], axis=1)

amino_pct = protein_aminos.div(protein_aminos.protein,axis=0).iloc[:,1:]
amino_pct.dropna(how="all", inplace=True)

In [126]:
pca = PCA().set_output(transform="pandas")
pca.fit(amino_pct)
pca.explained_variance_
pca.explained_variance_ratio_
amino_comp_pca = pca.transform(amino_pct)
scaler = scale().set_output(transform="pandas")
amino_comp_pca_scaled = scaler.fit_transform(amino_comp_pca)

In [127]:
#keeping >90% of var expl
kmeans = KMeans(n_clusters = 2)
amino_comp_pca_scaled_principal = amino_comp_pca_scaled.iloc[:,0:4]
amino_comp_pca_scaled_principal_kmclass = pd.Series(kmeans.fit_predict(
                                            amino_comp_pca_scaled_principal))
amino_comp_pca_scaled_principal_kmclass.index = \
                                        amino_comp_pca_scaled_principal.index
amino_comp_pca_scaled_principal_kmclass.sort_values()

/Users/williammohr/opt/anaconda3/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


food
green_peas_cooked                           0
cheese_grass_fed_cheddar_whole_milk         0
tempeh_cooked                               0
soybeans_cooked                             0
pinto_beans_cooked                          0
navy_beans_cooked                           0
lima_beans_cooked                           0
tofu_firm                                   0
kidney_beans_cooked                         0
garbanzo_beans_cooked                       0
dried_peas_split_cooked                     0
black_beans_cooked                          0
lentils_cooked                              0
pumpkin_seeds_dried_shelled                 0
scallops_steamed                            1
sardines_atlantic_canned                    1
salmon_wild_coho_broiled                    1
cod_pacific_fillet_baked                    1
yogurt_grass_fed_whole_milk                 1
turkey_pasture_raised_light_meat_roasted    1
lamb_grass_fed_lean_loin_roasted            1
chicken_pasture_raised_breast

for "high protein" foods with more than 10 grams per serv and >20% of calorie contribution from protein, PCA can effectively determine animal or plant source for the food. When restriction is relaxed to 6 grams protein per serv, cheese is classified with pulses.